In [1]:
"""
Generate K extended Barabási–Albert graphs,
compute a Kamada–Kawai layout for each, and save them to disk as GML files.
"""

'\nGenerate K extended Barabási–Albert graphs,\ncompute a Kamada–Kawai layout for each, and save them to disk as GML files.\n'

In [1]:
# %% imports and parameters
import os
import random
from pathlib import Path
from datetime import datetime
import networkx as nx
from tqdm.auto import tqdm

In [27]:
# ---- user-tunable parameters ---------------------------------------------
K_GRAPHS = 1000                # how many graphs to generate
N_MIN, N_MAX = 50, 150       # vertex range (inclusive)
M_MIN, M_MAX = 1, 3         # edges to attach per new node in BA model
OUT_DIR  = Path("temp_BA")
SEED     = 42                # global RNG seed for reproducibility
# --------------------------------------------------------------------------
random.seed(SEED)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [28]:
# %% helper
def generate_extended_ba_graph(n: int, m: int, seed: int = None) -> nx.Graph:
    """Return a *connected* Barabási–Albert graph with `n` nodes and `m` edges per new node."""
    p = random.uniform(0, 0.1)
    q = random.uniform(0, 0.2)
    G = nx.extended_barabasi_albert_graph(n=n, m=m, p=p, q=q, seed=seed)
    # Optional: tag the graph with metadata
    G.graph["Type"] = "Extended Barabasi Albert Graph"
    G.graph["n"] = n
    G.graph["m"] = m
    G.graph["p"] = p
    G.graph["q"] = q
    G.graph["created"] = datetime.now().isoformat()
    return G

In [29]:
# %% main loop
print(f"Generating {K_GRAPHS} graphs … (output dir = {OUT_DIR.resolve()})")

for idx in tqdm(range(1, K_GRAPHS + 1)):
    n = random.randint(N_MIN, N_MAX)
    m = random.randint(M_MIN, M_MAX)

    G = generate_extended_ba_graph(n, m, seed=random.randint(0, 1_000_000))
    while not nx.is_connected(G) or nx.is_planar(G):
        G = generate_extended_ba_graph(n, m, seed=random.randint(0, 1_000_000))

    assert nx.is_connected(G)
    assert not nx.is_planar(G)

    # Compute Kamada–Kawai layout
    pos = nx.kamada_kawai_layout(G)
    # nx.draw(G, pos)
    # print("Diameter", nx.diameter(G))
    # Store coordinates as node attributes
    for node, (x, y) in pos.items():
        G.nodes[node]["x"] = float(x)
        G.nodes[node]["y"] = float(y)

    out_path = OUT_DIR / f"ba_{idx:02d}_n{n}_m{m}.gml"
    if int(idx) % 100 == 0:
        print(" •", idx, out_path.name)
    nx.write_gml(G, out_path)

print("✔ Done. Files written:")

Generating 1000 graphs … (output dir = C:\Users\Timo\LRZ Sync+Share\PhD\Projects\RLGD\SmartGD\runner\temp_BA)


  0%|          | 0/1000 [00:00<?, ?it/s]

 • 100 ba_100_n107_m3.gml
 • 200 ba_200_n77_m3.gml
 • 300 ba_300_n140_m2.gml
 • 400 ba_400_n111_m1.gml
 • 500 ba_500_n50_m3.gml
 • 600 ba_600_n105_m3.gml
 • 700 ba_700_n78_m1.gml
 • 800 ba_800_n90_m2.gml
 • 900 ba_900_n116_m3.gml
 • 1000 ba_1000_n82_m3.gml
✔ Done. Files written:


In [17]:
# replace disconnected graphs
folder = Path(str(OUT_DIR)+"/data")
gml_files = sorted(folder.glob("*.gml"))

print(f"Checking {len(gml_files)} GML files in {folder.resolve()} …")

for f in tqdm(gml_files):
    try:
        G = nx.read_gml(f).to_undirected()
    except Exception as e:
        print(f"[!] Could not read {f.name}: {e}")
        continue

    if nx.is_connected(G):
        continue  # keep the original file

    print(f"↺  {f.name} is *disconnected* → replacing")

    # delete the old file
    try:
        os.remove(f)
    except OSError as e:
        print(f"[!] Could not delete {f.name}: {e}")
        continue

    # generate a replacement (connected by construction)
    idx = f.name.split("_")[1]
    n = int(f.name.split("_")[2].removeprefix("n"))
    m = int(f.name.split("_")[3].removeprefix("m").removesuffix(".gml"))
    new_G = generate_extended_ba_graph(n, m)
    it = 1
    while not nx.is_connected(new_G):
        new_G = generate_extended_ba_graph(n, m)
        it+=1

    assert nx.is_connected(new_G)
    # compute Kamada-Kawai layout and store coords
    pos = nx.kamada_kawai_layout(new_G)
    for v, (x, y) in pos.items():
        new_G.nodes[v]["x"] = float(x)
        new_G.nodes[v]["y"] = float(y)

    out_path = OUT_DIR / f"data/ba_{idx}_n{n}_m{m}.gml"
    nx.write_gml(new_G, out_path)
    print(f"✔  wrote replacement ({n} nodes, m={m}) → {out_path.name} in {it} iterations")

gml_files = sorted(folder.glob("*.gml"))
print(f"Now has {len(gml_files)} GML files in {folder.resolve()} …")

Checking 10000 GML files in C:\Users\Timo\LRZ Sync+Share\PhD\Projects\RLGD\SmartGD\runner\extended_BA\data …


  0%|          | 0/10000 [00:00<?, ?it/s]

↺  ba_01_n377_m1.gml is *disconnected* → replacing
✔  wrote replacement (377 nodes, m=1) → ba_01_n377_m1.gml in 4 iterations
↺  ba_06_n192_m1.gml is *disconnected* → replacing
✔  wrote replacement (192 nodes, m=1) → ba_06_n192_m1.gml in 4 iterations
↺  ba_08_n102_m1.gml is *disconnected* → replacing
✔  wrote replacement (102 nodes, m=1) → ba_08_n102_m1.gml in 2 iterations
↺  ba_1000_n401_m1.gml is *disconnected* → replacing
✔  wrote replacement (401 nodes, m=1) → ba_1000_n401_m1.gml in 17 iterations
↺  ba_1006_n484_m1.gml is *disconnected* → replacing
✔  wrote replacement (484 nodes, m=1) → ba_1006_n484_m1.gml in 19 iterations
↺  ba_100_n393_m1.gml is *disconnected* → replacing
✔  wrote replacement (393 nodes, m=1) → ba_100_n393_m1.gml in 9 iterations
↺  ba_1024_n269_m1.gml is *disconnected* → replacing
✔  wrote replacement (269 nodes, m=1) → ba_1024_n269_m1.gml in 1 iterations
↺  ba_1029_n473_m1.gml is *disconnected* → replacing
✔  wrote replacement (473 nodes, m=1) → ba_1029_n473_m1.

In [30]:
from pathlib import Path
import random
import shutil

def split_graph_folder(
    src: str | Path,
    dst_train: str | Path,
    dst_test: str | Path,
    ratio: float = 0.8,
    pattern: str = "*.gml",
    seed: int | None = 42,
    move: bool = False,
) -> None:
    """
    Split all graph files in *src* into two folders with a `ratio` share.

    Parameters
    ----------
    src        : directory that already contains the graphs
    dst_train  : folder where the first (e.g. 80 %) chunk is copied/moved
    dst_test   : folder where the remaining (e.g. 20 %) chunk is copied/moved
    ratio      : fraction that goes into *dst_train*  (default 0.8 → 80 %)
    pattern    : glob pattern that selects the graph files (default `"*.gml"`)
    seed       : optional RNG seed to make the split reproducible
    move       : if True use **move** (`shutil.move`) instead of copy (`shutil.copy2`)
    """
    src = Path(src)
    dst_train = Path(dst_train)
    dst_test = Path(dst_test)

    if not src.is_dir():
        raise FileNotFoundError(f"Source folder {src} does not exist")

    # dst_train.mkdir(parents=True, exist_ok=True)
    # dst_test.mkdir(parents=True,  exist_ok=True)

    files = sorted(src.glob(pattern))
    if not files:
        raise RuntimeError(f"No files matching {pattern!r} in {src}")

    if seed is not None:
        random.seed(seed)
    random.shuffle(files)

    k = int(round(len(files) * ratio))
    train_files, test_files = files[:k], files[k:]

    with open(dst_train, "w") as dst:
        dst.write("\n".join([f.name for f in train_files]))
    with open(dst_test, "w") as dst:
        dst.write("\n".join([f.name for f in test_files]))
    # print([f.name for f in train_files], file=open(dst_train, "w"))
    # print([f.name for f in test_files], file=open(dst_test, "w"))






In [31]:
# copy (keep originals)
OUT_DIR = "../../sng/graphs/extended_BA_filtered2"
split_graph_folder(
    src=str(OUT_DIR)+"/data",
    dst_train=str(OUT_DIR)+"/train.txt",
    dst_test=str(OUT_DIR)+"/test.txt",
    ratio=1.0,          # 80 / 20 split
    pattern="*.gml",    # or *.graphml, *.json …
)

